In [238]:
import numpy as np
import jax
import jax.numpy as jnp

import pickle
from pathlib import Path

from train import load_data
from test_model import load_config
from models import HybridODE


### First load all the data into a library

In [239]:
angle_lib = list(range(0, 21, 5))
data_lib = []
for angle in angle_lib:
    base_name = "data" + str(angle) + "deg"
    data = np.concat([data for data in load_data(processed_dir="processed_data", base_name=base_name)])
    data_lib.append(data)

### Load the trained basis models


In [255]:
all_models = []
for i in angle_lib:  # just use 0 and 20 deg models as basis
    config = load_config("config.yaml")
    base_name = "data" + str(i) + "deg"
    config["data"]["input_dir"] = "isaac_data/" + base_name
    params_path = Path("results") / base_name / "model_params.pkl"   
    with open(params_path, "rb") as fp:
        params = pickle.load(fp)    
    model = HybridODE(config)
    all_models.append([model, params])

basis = [all_models[0], all_models[-1]]
n_basis = len(basis)


Using basic MLP model
Using basic MLP model
Using basic MLP model
Using basic MLP model
Using basic MLP model


### Perform linear regression

In [261]:
# Load some necessary info from config
st_dim = 7
dt = config['data']['dt']
neural_states = config['model']['neural_states']
nz = len(neural_states)

# --- helper: one basis evaluation ---
def one_basis(b, initial_state, current_input, next_input):
    model, params = basis[b]
    next_state = model.rk4_step(
        initial_state, current_input, next_input, dt, params, training=False
    )
    return next_state[jnp.array(neural_states)]  # shape (nz,)

def compute_basis(initial_state, current_input, next_input):
    # loop in Python because basis is a Python list
    z_list = []
    for model, params in basis:
        next_state = model.rk4_step(
            initial_state, current_input, next_input, dt, params, training=False
        )
        z_list.append(next_state[jnp.array(neural_states)])
    return jnp.stack(z_list, axis=1)  # shape (nz, n_basis)

# --- process one sample ---
def process_one_sample(data_slice):
    initial_state = data_slice[:st_dim, 0]
    next_state = data_slice[:st_dim, 1]
    current_input = data_slice[st_dim:, 0]
    next_input = data_slice[st_dim:, 1]

    z_real = next_state[jnp.array(neural_states)]  # shape (nz,)
    z_basis = compute_basis(initial_state, current_input, next_input)  # (nz, n_basis)
    return z_basis, z_real

# --- vectorize over all samples ---
process_all = jax.vmap(process_one_sample)
# --- jit compile ---
process_all = jax.jit(process_all)

In [270]:
data = data_lib[1]  # use 5 deg data as example
n_samples = data.shape[0]
Z_basis, Z_real = process_all(data)  # or [:1000]

Theta = []
for n in range(nz):
    X = Z_basis[:, n, :]   # (N, 2)
    y = Z_real[:, n]       # (N,)
    # closed-form least squares: theta = (X^T X)^(-1) X^T y
    theta = jnp.linalg.inv(X.T @ X) @ (X.T @ y)
    Theta.append(theta)

    # prediction
    y_pred = X @ theta  # shape (N,)

    # mean squared error
    mse = jnp.mean((y - y_pred) ** 2)

    # root mean squared error
    rmse = jnp.sqrt(mse)

    # mean absolute error
    mae = jnp.mean(jnp.abs(y - y_pred))
    # Print error metrics
    print("Errors in fit")
    print("RMSE:", rmse)
    print("MAE:", mae)

Theta = jnp.stack(Theta, axis=0)  # (nz, n_basis)
print("Coefficients:", Theta)

Errors in fit
RMSE: 0.06360524
MAE: 0.030300263
Errors in fit
RMSE: 0.1394038
MAE: 0.027766945
Errors in fit
RMSE: 0.09638777
MAE: 0.042114746
Coefficients: [[ 0.6125488   0.38891602]
 [ 0.90455246 -0.06664276]
 [ 0.6993408   0.30297852]]
